# Preprocessing Pipeline: Climate-Socioeconomic Panel

**Objective**: Transform cleaned EDA data into modeling-ready format with engineered features, proper scaling, and time-aware train/test split.

**Inputs**: 
- `../data/processed/df_cleaned.csv` (output from EDA notebook)

**Outputs**:
- `X_train`, `X_test`, `y_train`, `y_test` (saved to `../data/processed/`)
- `scaler.pkl`, `feature_names.json` (saved to `../models/`)
- Engineered feature documentation

**Key Steps**:
1. Load cleaned data
2. Implement feature engineering (per-capita metrics, lags, ratios, interactions)
3. Handle scaling with RobustScaler
4. TimeSeriesSplit: train on 1900-2009, test on 2010-2023
5. Save artifacts for reproducibility

**Note**: All transformations must be fit on training data only to avoid leakage.

In [1]:
# IMPORTS & CONFIGURATION 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
import os

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths
DATA_CLEAN = '../data/processed/df_cleaned.csv'
DATA_PROCESSED = '../data/processed/'
MODELS_DIR = '../models/'
FIGURES_DIR = '../outputs/figures/'

# Ensure output directories exist
for dir_path in [DATA_PROCESSED, MODELS_DIR, FIGURES_DIR]:
    os.makedirs(dir_path, exist_ok=True)

print("Imports and paths configured successfully.")

Imports and paths configured successfully.


In [ ]:
# LOAD DATA & INITIAL SETUP 
# Load cleaned dataset
df = pd.read_csv(DATA_CLEAN)
# Basic verification
print("Loaded dataset info:")
print(f"Shape: {df.shape}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print(f"Countries: {df['Country'].nunique()}")

TARGETS = ['Temperature_Anomaly', 'CO2_Emissions']

# Sort by Country and Year (CRITICAL for correct lag/rolling calculations)
df = df.sort_values(['Country', 'Year']).reset_index(drop=True)

# Temporal split to prevent future data leakage
SPLIT_YEAR = 2010
df_train = df[df['Year'] < SPLIT_YEAR].copy()
df_test = df[df['Year'] >= SPLIT_YEAR].copy()

print(f"\nTemporal split (Year < {SPLIT_YEAR}):")
print(f"Train set: {len(df_train)} rows | Years: {df_train['Year'].min()}-{df_train['Year'].max()}")
print(f"Test set:  {len(df_test)} rows | Years: {df_test['Year'].min()}-{df_test['Year'].max()}")

Loaded dataset info:
Shape: (23797, 26)
Year range: 1900 - 2023
Countries: 195

Temporal split (Year < 2010):
Train set: 21106 rows | Years: 1900-2009
Test set:  2691 rows | Years: 2010-2023


In [5]:
#FEATURE ENGINEERING

def create_base_features(data):
    """Create per-capita, ratio, and interaction features."""
    df_eng = data.copy()
    
    # Per-capita metrics
    df_eng['GDP_per_Capita'] = df_eng['GDP'] / (df_eng['Population'] + 1e-6)
    df_eng['CO2_per_Capita'] = df_eng['CO2_Emissions'] / (df_eng['Population'] + 1e-6)
    
    # Energy transition ratio
    df_eng['Renewable_Ratio'] = df_eng['Renewable_Energy_Usage'] / (
        df_eng['Fossil_Fuel_Usage'] + df_eng['Renewable_Energy_Usage'] + 1e-6
    )
    
    # Policy-economic interaction
    df_eng['Policy_GDP_Interaction'] = df_eng['Policy_Score'] * df_eng['GDP_per_Capita']
    
    # Decade indicator for structural trends
    df_eng['Decade'] = (df_eng['Year'] // 10) * 10
    
    return df_eng

# Apply base features to full dataset first
df = create_base_features(df)

# Add time-series features (per country)
lag_rolling_cols = ['GDP', 'CO2_Emissions', 'Renewable_Energy_Usage', 'Fossil_Fuel_Usage']
new_cols = []

for col in lag_rolling_cols:
    # Lags
    df[f'{col}_lag1'] = df.groupby('Country')[col].shift(1)
    df[f'{col}_lag5'] = df.groupby('Country')[col].shift(5)
    
    # Rolling statistics (5-year window)
    df[f'{col}_roll5_mean'] = df.groupby('Country')[col].transform(lambda x: x.rolling(5, min_periods=1).mean())
    df[f'{col}_roll5_std'] = df.groupby('Country')[col].transform(lambda x: x.rolling(5, min_periods=1).std())
    
    new_cols.extend([f'{col}_lag1', f'{col}_lag5', f'{col}_roll5_mean', f'{col}_roll5_std'])

# Remove rows with NaNs from lag operations (affects earliest years per country)
df = df.dropna(subset=new_cols).reset_index(drop=True)

# Report
print(f"Feature engineering complete.")
print(f"Original columns: 26")
print(f"New engineered columns: {len(new_cols) + 4}")  # +4 for base features
print(f"Final shape after lag filtering: {df.shape}")
print(f"Total columns after engineering: {df.shape[1]}")
print(f"Columns: {df.columns.tolist()}")

Feature engineering complete.
Original columns: 26
New engineered columns: 20
Final shape after lag filtering: (20872, 47)
Total columns after engineering: 47
Columns: ['Country', 'Year', 'Air_Pollution_Index', 'Arctic_Ice_Extent', 'Average_Rainfall', 'Average_Temperature', 'Biodiversity_Index', 'CO2_Emissions', 'Deforestation_Rate', 'Energy_Consumption_Per_Capita', 'Extreme_Weather_Events', 'Forest_Area', 'Fossil_Fuel_Usage', 'GDP', 'Industrial_Activity', 'Methane_Emissions', 'Ocean_Acidification', 'Per_Capita_Emissions', 'Policy_Score', 'Population', 'Renewable_Energy_Usage', 'Sea_Level_Rise', 'Solar_Energy_Potential', 'Temperature_Anomaly', 'Urbanization', 'Waste_Management', 'GDP_per_Capita', 'CO2_per_Capita', 'Renewable_Ratio', 'Policy_GDP_Interaction', 'Decade', 'GDP_lag1', 'GDP_lag5', 'GDP_roll5_mean', 'GDP_roll5_std', 'CO2_Emissions_lag1', 'CO2_Emissions_lag5', 'CO2_Emissions_roll5_mean', 'CO2_Emissions_roll5_std', 'Renewable_Energy_Usage_lag1', 'Renewable_Energy_Usage_lag5', '